In [19]:
import requests
import pandas as pd

# Layer 0 endpoint
url_layer0 = "https://services2.arcgis.com/utBig56dGaw25bwu/ArcGIS/rest/services/MSP_MHSO_Crash_Layer/FeatureServer/0/query"

crashes = []
offset = 0

print("Downloading main crash data (Fatal & Injury)...")

while True:
    payload = {
        "where": "Reporttype IN ('Fatal Crash', 'Injury Crash', 'FATAL CRASH', 'INJURY CRASH')",
        "outFields": "*",
        "f": "json",
        "returnGeometry": "true",
        "outSR": "4326",
        "resultOffset": str(offset),
        "resultRecordCount": "2000",
        "orderByFields": "OBJECTID ASC",
    }
    
    response = requests.post(url_layer0, data=payload)
    res = response.json()
    
    if "error" in res:
        print("ArcGIS Error:", res["error"])
        break
        
    features = res.get("features", [])
    if not features:
        break
        
    for feat in features:
        row = feat.get("attributes", {})
        geom = feat.get("geometry", {})
        if geom and "x" in geom and "y" in geom:
            row["longitude"] = geom["x"]
            row["latitude"] = geom["y"]
        crashes.append(row)
        
    print(f"Layer 0: Fetched {len(features)} records (Total: {len(crashes)})...")
    
    if len(features) < 2000:
        break
    offset += len(features)
df_crashes = pd.DataFrame(crashes)
df_crashes.to_csv("msp_crashes_main_filtered.csv", index=False)
print(f"Saved {len(df_crashes)} filtered records to msp_crashes_main_filtered.csv")

# Extract unique Reportnumber values for subtable query
report_ids = [str(x) for x in df_crashes["Reportnumber"].dropna().unique() if str(x).strip()]
print(f"Extracted {len(report_ids)} unique Reportnumber values.\n")

Layer 0: Fetched 2000 records (Total: 2000)...
Layer 0: Fetched 2000 records (Total: 4000)...
Layer 0: Fetched 2000 records (Total: 6000)...
Layer 0: Fetched 2000 records (Total: 8000)...
Layer 0: Fetched 2000 records (Total: 10000)...
Layer 0: Fetched 2000 records (Total: 12000)...
Layer 0: Fetched 2000 records (Total: 14000)...
Layer 0: Fetched 2000 records (Total: 16000)...
Layer 0: Fetched 2000 records (Total: 18000)...
Layer 0: Fetched 2000 records (Total: 20000)...
Layer 0: Fetched 2000 records (Total: 22000)...
Layer 0: Fetched 2000 records (Total: 24000)...
Layer 0: Fetched 2000 records (Total: 26000)...
Layer 0: Fetched 2000 records (Total: 28000)...
Layer 0: Fetched 2000 records (Total: 30000)...
Layer 0: Fetched 2000 records (Total: 32000)...
Layer 0: Fetched 2000 records (Total: 34000)...
Layer 0: Fetched 2000 records (Total: 36000)...
Layer 0: Fetched 2000 records (Total: 38000)...
Layer 0: Fetched 2000 records (Total: 40000)...
Layer 0: Fetched 2000 records (Total: 42000)

In [20]:

subtables = {
    "vehicles": "1",
    "passengers": "2",
    "nonmotorists": "3",
    "drivers": "4",
    "circumstances": "5",
}

chunk_size = 2000

for name, layer_id in subtables.items():
    print(f"Starting {name} (Layer {layer_id})...")
    sub_records = []
    
    for i in range(0, len(report_ids), chunk_size):
        chunk = report_ids[i : i + chunk_size]
        formatted_ids = ", ".join([f"'{cid}'" for cid in chunk])
        where_clause = f"Reportnumber IN ({formatted_ids})"
        
        offset = 0
        while True:
            sub_url = f"https://services2.arcgis.com/utBig56dGaw25bwu/ArcGIS/rest/services/MSP_MHSO_Crash_Layer/FeatureServer/{layer_id}/query"
            
            sub_payload = {
                "where": where_clause,
                "outFields": "*",
                "f": "json",
                "resultOffset": str(offset),
                "resultRecordCount": "2000",
                "orderByFields": "OBJECTID ASC",
            }
            
            res = requests.post(sub_url, data=sub_payload).json()
            features = res.get("features", [])
            if not features:
                break
                
            for feat in features:
                sub_records.append(feat.get("attributes", {}))
                
            if len(features) < 2000:
                break
            offset += len(features)

    df_sub = pd.DataFrame(sub_records)
    df_sub.to_csv(f"msp_{name}_filtered.csv", index=False)
    print(f"Saved {len(df_sub)} records to msp_{name}_filtered.csv\n")

print("Finished downloading all crash and subtable data.")

Starting vehicles (Layer 1)...
Saved 341710 records to msp_vehicles_filtered.csv

Starting passengers (Layer 2)...
Saved 137606 records to msp_passengers_filtered.csv

Starting nonmotorists (Layer 3)...
Saved 21937 records to msp_nonmotorists_filtered.csv

Starting drivers (Layer 4)...
Saved 329521 records to msp_drivers_filtered.csv

Starting circumstances (Layer 5)...


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [50]:
crashes = pd.read_csv("msp_crashes_main_filtered.csv")
vehicles = pd.read_csv("msp_vehicles_filtered.csv")
nonmotorists = pd.read_csv("msp_nonmotorists_filtered.csv")
drivers = pd.read_csv("msp_drivers_filtered.csv")
passengers = pd.read_csv("msp_passengers_filtered.csv")
severe_nonmo = nonmotorists[nonmotorists['NonMotoristInjurySeverity'].isin(["Suspected Serious Injury", "Fatal Injury"])]
severe_dri = drivers[drivers['DriverInjurySeverity'].isin(["Suspected Serious Injury", "Fatal Injury"])]
severe_pass = passengers[passengers['PassengerInjurySeverity'].isin(["Suspected Serious Injury", "Fatal Injury"])]
severe_crash_set = set(list(severe_nonmo['Reportnumber'].unique()) + list(severe_pass['Reportnumber'].unique()) + list(severe_dri['Reportnumber'].unique()))

/var/folders/0r/74g1680s6rz9r8mbc5tx0_0c0000gn/T/ipykernel_27667/167780742.py:1: DtypeWarning: Columns (0: MaintenanceLocation, 1: MaintenanceWorkersPresent, 2: Offroaddescription, 3: Trafficcontrolfunctioning) have mixed types. Specify dtype option on import or set low_memory=False.
  crashes = pd.read_csv("msp_crashes_main_filtered.csv")


In [54]:
severe_crashes = crashes[crashes['Reportnumber'].isin(severe_crash_set)]
severe_drivers = drivers[drivers['Reportnumber'].isin(severe_crash_set)]
severe_passengers = passengers[passengers['Reportnumber'].isin(severe_crash_set)]
severe_nonmotorists = nonmotorists[nonmotorists['Reportnumber'].isin(severe_crash_set)]
severe_crashes.to_csv("crashes.csv")
severe_drivers.to_csv("drivers.csv")
severe_passengers.to_csv("passengers.csv")
severe_nonmotorists.to_csv("nonmotorists.csv")

In [52]:
for x in severe_crashes.index:
    

,OBJECTID,Reportnumber,AgencyCode,CountyOfCrash,Reporttype,UseData,Crashdate,CrashPoint,Latitude,Longitude,Collisiontype,Doctype,Fixedobjectstruck,Harmfuleventone,Harmfuleventtwo,Interchangearea,Intersectiontype,Junction,Lanedirection,Lanenumber,Lanetype,Lighting,Logmile_Dir,MaintenanceClosure,MaintenanceLocation,MaintenanceWorkersPresent,MaintenanceZone,Milepoint,Milepointdirection,Milepointdistance,Milepointdistanceunits,Nontraffic,Numberoflanes,Offroaddescription,Reference_Roadname,Reference_Route_Number,Reference_Route_Type,Road_Name,Roadalignment,Roadcondition,Roaddivision,Roadgrade,Route_Number,Route_Type,Schoolbusinvolvement,Status_Id,Surfacecondition,Timeofcrash,Trafficcontrol,Trafficcontrolfunctioning,Weather,CrashYear,CrashMonth,CrashWeekday,CrashHour,VehicleCount,DriverCount,PassengerCount,NonMotoristCount,TotalPersons,DriverFatals,PassengerFatals,NonMotoristFatals,TotalFatals,DriverInjs,PassengerInjs,NonMotoristInjs,TotalInjs,FlagMotorc,FlagBike,FlagPedes,FlagSpeed,FlagAgg,FlagImp,FlagDist,FlagMature,FlagYoung,FlagUnbelt,DriverMotorcycleFatals,PassengerMotorcycleFatals,TotalMotorcycleFatals,DriverUnbeltedFatals,PassengerUnbeltedFatals,TotalUnbeltedFatals,MatureDriverFatals,YoungDriverFatals,PedestrianFatals,BicyclistFatals,CrashDateQuery,longitude,latitude
0,1,AE4320003G,Baltimore County Police,Baltimore,Fatal Crash,True,2018-08-12,Point,39.426859,-76.581408,Single Vehicle,ACRS,Curb,Fixed Object,Fixed Object,NaN,NaN,NaN,E,1.0,NaN,Dark No Lights,E,NaN,NaN,NaN,N,1.940,E,244.0,F,N,2.0,NaN,AINTREE RD,3841.0,CO,SEMINARY AVE,Curve Right,No Defects,"Two-Way, Not Divided",Grade Downhill,995.0,County Road,0,Approved,Dry,01:13:00,0,NaN,Clear,2018,8,0,1,1,1,0,0,1,1,0,0,1,0,0,0,0,0,0,0,0,0,1,1,0,1,1,0,0,0,1,0,1,0,1,0,0,1534032000000,-76.581408,39.426859
1,2,AE4852003P,Baltimore County Police,Baltimore,Fatal Crash,True,2018-09-17,Point,39.353697,-76.792177,Head On,ACRS,NaN,Other Vehicle,NaN,NaN,NaN,Non-Intersection,E,1.0,NaN,Daylight,S,NaN,NaN,NaN,N,0.486,S,139.0,F,N,2.0,NaN,SUMMIT AVE,7068.0,OP,WINDSOR MILL RD,Straight,No Defects,"Two-Way, Not Divided",Level,1900.0,County Road,0,Approved,Wet,17:02:00,1,NaN,Raining,2018,9,1,17,2,2,3,0,5,0,1,0,1,2,2,0,4,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,1537142400000,-76.792177,39.353697
2,3,AE5141003F,Baltimore County Police,Baltimore,Fatal Crash,True,2018-08-04,Point,39.357156,-76.767673,Other,ACRS,NaN,Pedestrian,Pedestrian,NaN,NaN,Non-Intersection,E,NaN,Left Turn Lane,Dark Lights On,E,NaN,NaN,NaN,N,7.147,W,232.0,F,N,4.0,NaN,ST JAMES RD,2077.0,CO,LIBERTY RD,Straight,No Defects,"Two-Way, Not Divided with a Continuous Left Turn",Level,26.0,Maryland Route,0,Approved,Dry,23:33:00,1,NaN,Clear,2018,8,6,23,2,2,0,1,3,0,0,1,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1533340800000,-76.767673,39.357156
3,4,ADD721006T,Baltimore City Police,Baltimore City,Fatal Crash,True,2018-09-08,Point,39.285736,-76.652483,Single Vehicle,ACRS,NaN,Pedestrian,NaN,NaN,NaN,Non-Intersection,N,1.0,NaN,Daylight,NaN,NaN,NaN,NaN,N,0.000,N,300.0,F,N,1.0,NaN,FREDERICK AVENUE,NaN,NaN,MCPHAIL STREET,Straight,Holes Ruts Etc,"Two-Way, Not Divided",Level,NaN,NaN,0,Approved,Wet,16:19:00,1,NaN,Raining,2018,9,6,16,1,1,0,1,2,0,0,1,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,1536364800000,-76.652483,39.285736
4,5,AE3568003H,Baltimore County Police,Baltimore,Fatal Crash,True,2018-08-29,Point,39.240642,-76.680369,Head On Left Turn,ACRS,NaN,Other Vehicle,NaN,NaN,T-Intersection,Intersection,N,2.0,NaN,Daylight,N,NaN,NaN,NaN,N,1.190,N,0.0,F,N,4.0,NaN,MONUMENTAL RD,3178.0,CO,WASHINGTON BLVD,Straight,No Defects,"Two-Way, Not Divided",Hill Uphill,1.0,United States Route,0,Approved,Dry,14:30:00,6,NaN,Cloudy,2018,8,3,14,2,2,0,0,2,1,0,0,1,0,0,0,0,1,0,0,1,0,0,1,1,1,0,1,0,1,0,0,0,0,1,0,0,1535500800000,-76.680369,39.240642
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,.

### passengers['PassengerInjurySeverity'].unique()

In [ ]:
drivers